# DataHub Error Case 05 — Elasticsearch Down

**Monitor:** DataHub Elasticsearch Error  
**Component:** datahub-gms  
**Scenario:** opensearch-cluster-master pod shutdown → GMS loses ES connection

## Flow
```
opensearch-cluster-master pod scaled to 0
  → GMS ESSearchDAO tries to query → Timeout connecting
  → ESSearchDAO ERROR: 'Search query failed'
  → SearchResolver ERROR: 'Failed to execute search'
  → GraphQLController ERROR: 'Errors while executing query'
  → Client receives: {errors: [{code: 500, type: SERVER_ERROR}]}
```

## ⚠️ WARNING
```
DataHub search will be unavailable during this test (~2-3 min)
All indices preserved — no data loss
```

In [8]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *
import urllib3
urllib3.disable_warnings()

r = health_check()
print(f'GMS health: {r.status_code}')
session = get_session()
print('Logged in')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
GMS health: 200
Logged in


In [9]:
# Cell 2: Check DataHub indices via LoadBalancer
# opensearch-cluster-master-lb: 192.168.139.2:9200
r = requests.get(
    f'{DATAHUB_ES}/_cat/indices?h=index,health,docs.count&s=index'
)
print('DataHub OpenSearch indices:')
print(r.text)

DataHub OpenSearch indices:
.opendistro-job-scheduler-lock                           yellow   1
.opensearch-observability                                green    0
.plugins-ml-config                                       green    1
applicationindex_v2                                      yellow   0
assertion_assertionruneventaspect_v1                     yellow   0
assertionindex_v2                                        yellow   0
businessattributeindex_v2                                yellow   0
chart_chartusagestatisticsaspect_v1                      yellow   0
chartindex_v2                                            yellow   0
containerindex_v2                                        yellow   0
corpgroupindex_v2                                        yellow   0
corpuserindex_v2                                         yellow   1
dashboard_dashboardusagestatisticsaspect_v1              yellow   0
dashboardindex_v2                                        yellow   0
datacontractindex_v2

In [10]:
# Cell 3: Verify ES healthy + search works before test
r = requests.get(f'{DATAHUB_ES}/_cluster/health')
print(f'ES cluster health: {r.status_code} → {r.json().get("status")}')

r2 = graphql(
    '{ search(input: { type: DATASET, query: "test", start: 0, count: 1 }) { total } }',
    session=session
)
print(f'Search before shutdown: {r2.status_code} → {r2.json().get("data", {})}')

ES cluster health: 200 → yellow
Search before shutdown: 200 → {'search': None}


In [11]:
# Cell 4: Shutdown opensearch-cluster-master
print('Scaling opensearch-cluster-master to 0...')
kubectl('scale statefulset opensearch-cluster-master --replicas=0 -n datahub-hynix')
print('Waiting 15s for ES to go down...')
import time; time.sleep(15)

try:
    r = requests.get(f'{DATAHUB_ES}/_cluster/health', timeout=3)
    print(f'ES still reachable: {r.status_code}')
except Exception as e:
    print(f'ES unreachable: {e}')

Scaling opensearch-cluster-master to 0...
statefulset.apps/opensearch-cluster-master scaled

Waiting 15s for ES to go down...
ES unreachable: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [12]:
# Cell 5: Trigger search → Timeout connecting → ERROR chain
r = graphql(
    '{ search(input: { type: DATASET, query: "test", start: 0, count: 5 }) { total } }',
    session=session
)
print(f'Search during ES down: {r.status_code}')
resp = r.json()
if 'errors' in resp:
    err = resp['errors'][0]
    print(f'Error type: {err.get("extensions", {}).get("type")}')
    print(f'Message: {err.get("message", "")[:150]}')
else:
    print(json.dumps(resp.get('data', {}), indent=2)[:200])

Search during ES down: 200
Error type: SERVER_ERROR
Message: java.lang.RuntimeException: Failed to execute search: entity type DATASET, query test, filters: null, orFilters: null, start: 0, count: 5, searchFlags


In [13]:
# Cell 6: Find Elasticsearch error logs in OpenSearch
# Using logger field match — message is too long for exact match
wait_for_logs(5)
client = os_client()
result = client.search(
    index=LOG_INDEX,
    body={
        'query': {
            'bool': {
                'must': [
                    {'range': {'@timestamp': {'gte': 'now-3m'}}},
                    {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                    {'term': {'level.keyword': 'ERROR'}},
                ],
                'should': [
                    {'match': {'logger': 'ESSearchDAO'}},
                    {'match': {'logger': 'SearchResolver'}},
                    {'match': {'logger': 'GraphQLController'}},
                    {'match': {'logger': 'DataHubAuthorizer'}},
                    {'match': {'message': 'Search query failed'}},
                    {'match': {'message': 'Timeout connecting'}},
                ],
                'minimum_should_match': 1,
            }
        },
        'sort': [{'@timestamp': {'order': 'desc'}}],
        'size': 5,
    }
)
hits = result['hits']['hits']
print(f'Found {len(hits)} error logs')
print_logs(hits)

# If still empty — check Fluent Bit is parsing logger field
if not hits:
    print('\n--- Debug: check logger field exists ---')
    r2 = client.search(
        index=LOG_INDEX,
        body={
            'query': {
                'bool': {
                    'must': [
                        {'range': {'@timestamp': {'gte': 'now-3m'}}},
                        {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                        {'term': {'level.keyword': 'ERROR'}},
                    ]
                }
            },
            'sort': [{'@timestamp': {'order': 'desc'}}],
            'size': 3,
        }
    )
    print(f'Any ERROR logs in last 3 min: {r2["hits"]["total"]["value"]}')
    print_logs(r2['hits']['hits'])

⏳ Waiting 5s for logs to reach OpenSearch...
Found 5 error logs
✅ Found 5 log entries:

  [2026-06-11T00:03:42] ERROR | datahub-gms | c.l.d.g.e.DataHubDataFetcherExceptionHandler
         Failed to execute

  [2026-06-11T00:03:42] ERROR | datahub-gms | c.l.m.s.e.query.ESSearchDAO
         Search query failed

  [2026-06-11T00:03:42] ERROR | datahub-gms | c.l.m.s.e.query.ESSearchDAO
         Response to the failed search query: null

  [2026-06-11T00:03:42] ERROR | datahub-gms | c.l.d.g.r.search.SearchResolver
         Failed to execute search: entity type DATASET, query test, filters: null, orFilters: null, start: 0, count: 5, searchFla

  [2026-06-11T00:02:39] ERROR | datahub-gms | c.d.authorization.DataHubAuthorizer
         Failed to retrieve policy urns! Skipping updating policy cache until next refresh. count: 1000, scrollId: null



In [15]:
# Cell 7: Restore — scale ES back up
print('Restoring opensearch-cluster-master...')
kubectl('scale statefulset opensearch-cluster-master --replicas=1 -n datahub-hynix')
import time
print('Waiting for ES to restart (~30s)...')
time.sleep(30)

try:
    r = requests.get(f'{DATAHUB_ES}/_cluster/health', timeout=5)
    print(f'ES health: {r.status_code} → {r.json().get("status")}')
except Exception as e:
    print(f'Not yet ready: {e}')

r2 = graphql(
    '{ search(input: { type: DATASET, query: "test", start: 0, count: 1 }) { total } }',
    session=session
)
print(f'Search after recovery: {r2.status_code} → {r2.json().get("data", {})}')

Restoring opensearch-cluster-master...
statefulset.apps/opensearch-cluster-master scaled

Waiting for ES to restart (~30s)...
ES health: 200 → yellow
Search after recovery: 200 → {'search': None}


In [14]:
print('\n=== Summary ===')
print('Error:   Elasticsearch connection timeout')
print('Cause:   opensearch-cluster-master pod down')
print('Monitor: DataHub Elasticsearch Error fires on ERROR count > 0 in 5 min')
print('Loggers:')
print('  c.l.m.s.e.query.ESSearchDAO       → Search query failed')
print('  c.l.d.g.r.search.SearchResolver   → Failed to execute search')
print('  c.datahub.graphql.GraphQLController → Errors while executing query')
print('Fix:     kubectl scale statefulset opensearch-cluster-master --replicas=1')
print('Note:    GMS auto-reconnects after ES restarts — no data loss')


=== Summary ===
Error:   Elasticsearch connection timeout
Cause:   opensearch-cluster-master pod down
Monitor: DataHub Elasticsearch Error fires on ERROR count > 0 in 5 min
Loggers:
  c.l.m.s.e.query.ESSearchDAO       → Search query failed
  c.l.d.g.r.search.SearchResolver   → Failed to execute search
  c.datahub.graphql.GraphQLController → Errors while executing query
Fix:     kubectl scale statefulset opensearch-cluster-master --replicas=1
Note:    GMS auto-reconnects after ES restarts — no data loss
